# Matrix Factorization

In [1]:
import numpy as np
import pandas as pd
from collections import defaultdict
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import random

# Configuration
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = Path("/content/drive/MyDrive/yuran_files")   # change if needed
TRAIN_PATH = BASE_PATH / "train_complete.csv"
VAL_PATH   = BASE_PATH / "val_complete.csv"
TEST_PATH  = BASE_PATH / "test_complete.csv"
MASTER_ANIME_PATH = BASE_PATH / "MASTER_ANIME_TOWER_FEATURES_1BASED.npy"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

Mounted at /content/drive
Using device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [20]:
# Hyperparameters (aligned with two_tower_bpr_modified)
N_FACTORS    = 20
LR           = 0.001
EPOCHS       = 5
WEIGHT_DECAY = 1e-5        # L2 regularisation
BATCH_SIZE   = 1024

EVAL_K         = 10
EVAL_MAX_USERS = 300       # cap for per-epoch val (speed); matches Two-Tower
POS_THRESHOLD  = 6.0       # lower from 8.0
EVAL_NEG_SAMPLES = 300

## Load Data

In [3]:
# Load train, val, test
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

if "score" in train_df.columns and "rating" not in train_df.columns:
    train_df = train_df.rename(columns={"score": "rating"})
if "score" in val_df.columns and "rating" not in val_df.columns:
    val_df = val_df.rename(columns={"score": "rating"})
if "score" in test_df.columns and "rating" not in test_df.columns:
    test_df = test_df.rename(columns={"score": "rating"})

required_cols = ["user_idx", "anime_idx", "rating"]
for c in required_cols:
    if c not in train_df.columns:
        raise ValueError(f"Missing column in train.csv: {c}")
    if c not in val_df.columns:
        raise ValueError(f"Missing column in val.csv: {c}")
    if c not in test_df.columns:
        raise ValueError(f"Missing column in test.csv: {c}")

train_df = train_df[required_cols].copy()
val_df = val_df[required_cols].copy()
test_df = test_df[required_cols].copy()

for df in [train_df, val_df, test_df]:
    df["user_idx"] = df["user_idx"].astype(int)
    df["anime_idx"] = df["anime_idx"].astype(int)
    df["rating"] = df["rating"].astype(float)

# Derive embedding sizes from the global max index (same logic as two-tower)
max_user_idx = max(train_df['user_idx'].max(), val_df['user_idx'].max(), test_df['user_idx'].max())
max_anime_idx = max(train_df['anime_idx'].max(), val_df['anime_idx'].max(), test_df['anime_idx'].max())

NUM_USERS_EMBEDDING = max_user_idx + 1
NUM_ITEMS_EMBEDDING = max_anime_idx + 1

print(f"Train: {train_df.shape}  Val: {val_df.shape}  Test: {test_df.shape}")
print(f"NUM_USERS_EMBEDDING: {NUM_USERS_EMBEDDING}")
print(f"NUM_ITEMS_EMBEDDING: {NUM_ITEMS_EMBEDDING}")

# Remove rating of 0
# train_df = train_df[train_df['rating'] > 0]
# val_df = val_df[val_df['rating'] > 0]
# test_df = test_df[test_df['rating'] > 0]

Train: (82782709, 3)  Val: (10347839, 3)  Test: (10347839, 3)
NUM_USERS_EMBEDDING: 292565
NUM_ITEMS_EMBEDDING: 13010


In [4]:
# Remove rating of 0
train_df = train_df[train_df['rating'] > 0]
val_df = val_df[val_df['rating'] > 0]
test_df = test_df[test_df['rating'] > 0]

print(f"Train: {train_df.shape}  Val: {val_df.shape}  Test: {test_df.shape}")

Train: (49742364, 3)  Val: (6217132, 3)  Test: (6215263, 3)


## Prepare Positives / Seen Maps

In [5]:
from collections import Counter

# Precompute lookup structures
user_seen_train = (
    train_df.groupby("user_idx")["anime_idx"].apply(set).to_dict()
)
test_pos_by_user = (
    test_df[test_df["rating"] >= POS_THRESHOLD]
    .groupby("user_idx")["anime_idx"].apply(set).to_dict()
)
train_pos_by_user = (
    train_df[train_df["rating"] >= POS_THRESHOLD]
    .groupby("user_idx")["anime_idx"].apply(set).to_dict()
)
train_pos_df = train_df[train_df["rating"] >= POS_THRESHOLD].copy()
val_pos_by_user = (
    val_df[val_df["rating"] >= POS_THRESHOLD]
    .groupby("user_idx")["anime_idx"].apply(set).to_dict()
)

## Dataset Classes

In [6]:
# Rating-regression dataset (used for MSE training)
class AnimeInteractionDataset(Dataset):
    def __init__(self, dataframe):
        self.users   = torch.tensor(dataframe["user_idx"].values,  dtype=torch.long)
        self.animes  = torch.tensor(dataframe["anime_idx"].values, dtype=torch.long)
        self.ratings = torch.tensor(dataframe["rating"].values,    dtype=torch.float32)

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        return self.users[idx], self.animes[idx], self.ratings[idx]


# BPR pairwise dataset (used for BPR training)
class MatrixFactorizationBPRDataset(Dataset):
    def __init__(self, pos_df, user_seen_map, num_items):
        self.users     = pos_df["user_idx"].to_numpy(np.int64)
        self.pos_items = pos_df["anime_idx"].to_numpy(np.int64)
        self.user_seen = user_seen_map
        self.num_items = num_items

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        u = int(self.users[idx])
        p = int(self.pos_items[idx])

        seen = self.user_seen.get(u, set())
        n = np.random.randint(1, self.num_items)
        while n in seen:
            n = np.random.randint(1, self.num_items)

        return (
            torch.tensor(u, dtype=torch.long),
            torch.tensor(p, dtype=torch.long),
            torch.tensor(n, dtype=torch.long)
        )


# DataLoaders
train_loader = DataLoader(
    AnimeInteractionDataset(train_df), batch_size=BATCH_SIZE, shuffle=True
)
val_loader = DataLoader(
    AnimeInteractionDataset(val_df), batch_size=BATCH_SIZE, shuffle=False
)
test_loader = DataLoader(
    AnimeInteractionDataset(test_df), batch_size=BATCH_SIZE, shuffle=False
)

bpr_train_loader = DataLoader(
    MatrixFactorizationBPRDataset(train_pos_df, user_seen_train, NUM_ITEMS_EMBEDDING),
    batch_size=BATCH_SIZE, shuffle=True
)

## Model Definitions

### Matrix Factorization Model with Biases

In [7]:
class MatrixFactorization(nn.Module):
    def __init__(self, num_users, num_items, n_factors=20):
        super().__init__()

        # Embeddings
        self.user_factors = nn.Embedding(num_users, n_factors)
        self.item_factors = nn.Embedding(num_items, n_factors)

        # Biases
        self.user_bias = nn.Embedding(num_users, 1)
        self.item_bias = nn.Embedding(num_items, 1)

        # Global bias
        self.global_bias = nn.Parameter(torch.zeros(1))

        # Init
        nn.init.normal_(self.user_factors.weight, std=0.01)
        nn.init.normal_(self.item_factors.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, user, item):
        p_u = self.user_factors(user)
        q_i = self.item_factors(item)

        dot = (p_u * q_i).sum(dim=1)

        b_u = self.user_bias(user).squeeze()
        b_i = self.item_bias(item).squeeze()

        return self.global_bias + b_u + b_i + dot

### Matrix Factorization without Biases

In [8]:
class MatrixFactorizationNoBias(nn.Module):
    def __init__(self, num_users, num_items, n_factors=20):
        super().__init__()

        # Embeddings only
        self.user_factors = nn.Embedding(num_users, n_factors)
        self.item_factors = nn.Embedding(num_items, n_factors)

        # Init
        nn.init.normal_(self.user_factors.weight, std=0.01)
        nn.init.normal_(self.item_factors.weight, std=0.01)

    def forward(self, user, item):
        p_u = self.user_factors(user)
        q_i = self.item_factors(item)

        # Dot product
        return (p_u * q_i).sum(dim=1)

## Evaluation Helper Functions

In [9]:
# Evaluation function to get RSME and MAE
def evaluate_regression(model, data_loader):
    model.eval()

    preds_list = []
    actuals_list = []

    with torch.no_grad():
        for users, items, ratings in data_loader:
            users = users.to(DEVICE)
            items = items.to(DEVICE)

            preds = model(users, items)

            preds_list.extend(preds.cpu().numpy())
            actuals_list.extend(ratings.numpy())

    preds = np.array(preds_list)
    actuals = np.array(actuals_list)

    rmse = np.sqrt(np.mean((preds - actuals) ** 2))
    mae = np.mean(np.abs(preds - actuals))

    return rmse, mae

In [10]:
def build_user_item_sets(df):
    user_items = defaultdict(set)
    for u, i in zip(df['user_idx'], df['anime_idx']):
        user_items[u].add(i)
    return user_items

In [11]:
# Full-catalog evaluation for Matrix Factorization (derived based on Two-Tower code)
def _dcg_at_k(binary_hits):
    if len(binary_hits) == 0:
        return 0.0
    denom = np.log2(np.arange(2, len(binary_hits) + 2))
    return float((binary_hits / denom).sum())

@torch.no_grad()
def evaluate_full_catalog_at_k(model, pos_by_user, seen_items_by_user, k=10, max_users=None):
    """
    Full-catalog evaluation for Matrix Factorization model.
    For each user, scores all items except seen training items, ranks them,
    and computes ranking metrics against ground-truth positives.

    max_users=None uses ALL users with test positives (slow but complete).
    max_users=300 matches Two-Tower's default cap.
    """
    if len(pos_by_user) == 0:
        return 0.0, 0.0, 0.0, 0.0

    model.eval()
    all_item_ids = np.arange(NUM_ITEMS_EMBEDDING)

    users = list(pos_by_user.keys())
    if max_users is not None and len(users) > max_users:
        np.random.seed(SEED)
        users = list(np.random.choice(users, size=max_users, replace=False))

    precisions, recalls, hitrates, ndcgs = [], [], [], []

    for u in users:
        u = int(u)
        pos_items = np.array(list(pos_by_user.get(u, set())), dtype=np.int64)
        if len(pos_items) == 0:
            continue

        seen = seen_items_by_user.get(u, set())
        candidates = np.setdiff1d(all_item_ids, list(seen))
        if len(candidates) == 0:
            continue

        u_tensor = torch.tensor([u] * len(candidates), dtype=torch.long).to(DEVICE)
        i_tensor = torch.tensor(candidates, dtype=torch.long).to(DEVICE)
        scores = model(u_tensor, i_tensor).cpu().numpy()

        top_idx = np.argpartition(scores, -k)[-k:]
        top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]
        top_items = candidates[top_idx]

        hits = np.isin(top_items, pos_items).astype(np.float32)
        num_hits = float(hits.sum())

        precisions.append(num_hits / k)
        recalls.append(num_hits / max(len(pos_items), 1))
        hitrates.append(1.0 if num_hits > 0 else 0.0)

        dcg = _dcg_at_k(hits)
        ideal_hits = np.ones(min(k, len(pos_items)), dtype=np.float32)
        idcg = _dcg_at_k(ideal_hits)
        ndcgs.append((dcg / idcg) if idcg > 0 else 0.0)

    if len(precisions) == 0:
        return 0.0, 0.0, 0.0, 0.0

    return (
        float(np.mean(precisions)),
        float(np.mean(recalls)),
        float(np.mean(hitrates)),
        float(np.mean(ndcgs))
    )

In [12]:
@torch.no_grad()
def evaluate_ranking_at_k(model, pos_by_user, seen_items_by_user, k=10, max_users=300, neg_samples=300):
    if len(pos_by_user) == 0:
        return 0.0, 0.0, 0.0, 0.0

    model.eval()
    users = list(pos_by_user.keys())

    if max_users is not None and len(users) > max_users:
        users = list(np.random.choice(users, size=max_users, replace=False))

    precisions, recalls, hitrates, ndcgs = [], [], [], []

    for u in users:
        u = int(u)
        pos_items = np.array(list(pos_by_user.get(u, set())), dtype=np.int64)
        if len(pos_items) == 0:
            continue

        seen = seen_items_by_user.get(u, set())
        neg = np.random.randint(1, NUM_ITEMS_EMBEDDING, size=neg_samples, dtype=np.int64)
        neg = np.array(
            [x for x in neg if (x not in seen and x not in pos_by_user.get(u, set()))],
            dtype=np.int64
        )
        if len(neg) == 0:
            continue

        candidates = np.unique(np.concatenate([pos_items, neg]))
        if len(candidates) == 0:
            continue

        u_tensor = torch.full((len(candidates),), u, dtype=torch.long, device=DEVICE)
        i_tensor = torch.tensor(candidates, dtype=torch.long, device=DEVICE)

        scores = model(u_tensor, i_tensor).detach().cpu().numpy()

        top_idx = np.argsort(scores)[-k:][::-1]
        top_items = candidates[top_idx]

        hits = np.isin(top_items, pos_items).astype(np.float32)
        num_hits = float(hits.sum())

        precisions.append(num_hits / k)
        recalls.append(num_hits / max(len(pos_items), 1))
        hitrates.append(1.0 if num_hits > 0 else 0.0)

        dcg = _dcg_at_k(hits)
        ideal_hits = np.ones(min(k, len(pos_items)), dtype=np.float32)
        idcg = _dcg_at_k(ideal_hits)
        ndcgs.append((dcg / idcg) if idcg > 0 else 0.0)

    if len(precisions) == 0:
        return 0.0, 0.0, 0.0, 0.0

    return (
        float(np.mean(precisions)),
        float(np.mean(recalls)),
        float(np.mean(hitrates)),
        float(np.mean(ndcgs))
    )

## Training Function

### MSE Training

In [21]:
def train_model(model, train_loader, val_loader, val_pos_by_user, user_seen_train):
    model.to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.MSELoss()

    for epoch in range(EPOCHS):
        model.train()
        train_losses = []

        for users, items, ratings in train_loader:
            users, items, ratings = users.to(DEVICE), items.to(DEVICE), ratings.to(DEVICE)

            optimizer.zero_grad()
            preds = model(users, items)
            loss = loss_fn(preds, ratings)

            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        # Val ranking metrics each epoch (matches Two-Tower's code's training output)
        val_p, val_r, val_hr, val_ndcg = evaluate_full_catalog_at_k(
            model=model,
            pos_by_user=val_pos_by_user,
            seen_items_by_user=user_seen_train,
            k=EVAL_K,
            max_users=EVAL_MAX_USERS   # 300
        )
        val_rmse, val_mae = evaluate_regression(model, val_loader)

        print(
            f"Epoch {epoch+1:02d} | "
            f"train_loss={np.mean(train_losses):.4f} | "
            f"Val RMSE={val_rmse:.4f} | "
            f"Val MAE={val_mae:.4f} | "
            f"Val P@{EVAL_K}={val_p:.4f} | "
            f"Val R@{EVAL_K}={val_r:.4f} | "
            f"Val HR@{EVAL_K}={val_hr:.4f} | "
            f"Val NDCG@{EVAL_K}={val_ndcg:.4f}"
        )
        print("-" * 80)

### BPR Training

In [16]:
import torch.nn.functional as F

def bpr_loss(pos_scores, neg_scores):
  return -F.logsigmoid(pos_scores - neg_scores).mean()
  # return -torch.mean(F.logsigmoid(pos_scores - neg_scores))


def train_model_bpr(model, bpr_train_loader, val_loader, val_pos_by_user, user_seen_train):
    model.to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    for epoch in range(EPOCHS):
        model.train()
        train_losses = []

        for users, pos_items, neg_items in bpr_train_loader:
            users     = users.to(DEVICE)
            pos_items = pos_items.to(DEVICE)
            neg_items = neg_items.to(DEVICE)

            optimizer.zero_grad()
            pos_scores = model(users, pos_items)
            neg_scores = model(users, neg_items)
            loss       = bpr_loss(pos_scores, neg_scores)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        # Validation ranking metrics (full catalog)
        val_p, val_r, val_hr, val_ndcg = evaluate_full_catalog_at_k(
            model=model,
            pos_by_user=val_pos_by_user,
            seen_items_by_user=user_seen_train,
            k=EVAL_K,
            max_users=EVAL_MAX_USERS
        )

        # Regression metrics are less meaningful for BPR but kept for reference
        val_rmse, val_mae = evaluate_regression(model, val_loader)

        print(
            f"Epoch {epoch+1:02d} | "
            f"bpr_loss={np.mean(train_losses):.4f} | "
            f"Val RMSE={val_rmse:.4f} | Val MAE={val_mae:.4f} | "
            f"Val P@{EVAL_K}={val_p:.4f} | Val R@{EVAL_K}={val_r:.4f} | "
            f"Val HR@{EVAL_K}={val_hr:.4f} | Val NDCG@{EVAL_K}={val_ndcg:.4f}"
        )
        print("-" * 90)

## Model Training

### 1. Matrix Factorization with Biases (MSE)

In [22]:
model = MatrixFactorization(NUM_USERS_EMBEDDING, NUM_ITEMS_EMBEDDING, N_FACTORS)

train_model(model, train_loader, val_loader, val_pos_by_user, user_seen_train)

Epoch 01 | train_loss=3.3471 | Val RMSE=1.3150 | Val MAE=0.9871 | Val P@10=0.0297 | Val R@10=0.0314 | Val HR@10=0.2300 | Val NDCG@10=0.0387
--------------------------------------------------------------------------------
Epoch 02 | train_loss=1.7079 | Val RMSE=1.3001 | Val MAE=0.9757 | Val P@10=0.0353 | Val R@10=0.0374 | Val HR@10=0.2633 | Val NDCG@10=0.0472
--------------------------------------------------------------------------------
Epoch 03 | train_loss=1.6771 | Val RMSE=1.2919 | Val MAE=0.9684 | Val P@10=0.0330 | Val R@10=0.0320 | Val HR@10=0.2400 | Val NDCG@10=0.0466
--------------------------------------------------------------------------------
Epoch 04 | train_loss=1.6626 | Val RMSE=1.2886 | Val MAE=0.9649 | Val P@10=0.0390 | Val R@10=0.0394 | Val HR@10=0.2867 | Val NDCG@10=0.0496
--------------------------------------------------------------------------------
Epoch 05 | train_loss=1.6560 | Val RMSE=1.2870 | Val MAE=0.9635 | Val P@10=0.0363 | Val R@10=0.0372 | Val HR@10=0.27

### 2. Matrix Factorization without Biases (MSE)

In [23]:
# Training matrix factorization with no bias model
model_no_bias = MatrixFactorizationNoBias(NUM_USERS_EMBEDDING, NUM_ITEMS_EMBEDDING, N_FACTORS)

train_model(model_no_bias, train_loader, val_loader, val_pos_by_user, user_seen_train)

Epoch 01 | train_loss=5.6497 | Val RMSE=1.7327 | Val MAE=1.2690 | Val P@10=0.0387 | Val R@10=0.0434 | Val HR@10=0.2867 | Val NDCG@10=0.0586
--------------------------------------------------------------------------------
Epoch 02 | train_loss=2.9282 | Val RMSE=1.6882 | Val MAE=1.2320 | Val P@10=0.0413 | Val R@10=0.0527 | Val HR@10=0.3000 | Val NDCG@10=0.0618
--------------------------------------------------------------------------------
Epoch 03 | train_loss=2.7831 | Val RMSE=1.6582 | Val MAE=1.2064 | Val P@10=0.0470 | Val R@10=0.0599 | Val HR@10=0.3400 | Val NDCG@10=0.0681
--------------------------------------------------------------------------------
Epoch 04 | train_loss=2.7244 | Val RMSE=1.6483 | Val MAE=1.1965 | Val P@10=0.0510 | Val R@10=0.0638 | Val HR@10=0.3567 | Val NDCG@10=0.0697
--------------------------------------------------------------------------------
Epoch 05 | train_loss=2.6929 | Val RMSE=1.6382 | Val MAE=1.1867 | Val P@10=0.0507 | Val R@10=0.0621 | Val HR@10=0.36

### 3. Matrix Factorization without Biases (BPR)

In [24]:
model_no_bias_bpr = MatrixFactorizationNoBias(NUM_USERS_EMBEDDING, NUM_ITEMS_EMBEDDING, N_FACTORS)

train_model_bpr(model_no_bias_bpr, bpr_train_loader, val_loader, val_pos_by_user, user_seen_train)

Epoch 01 | bpr_loss=0.3866 | Val RMSE=6.8289 | Val MAE=6.6181 | Val P@10=0.0813 | Val R@10=0.0771 | Val HR@10=0.4700 | Val NDCG@10=0.1026
------------------------------------------------------------------------------------------
Epoch 02 | bpr_loss=0.3672 | Val RMSE=6.8287 | Val MAE=6.6179 | Val P@10=0.0813 | Val R@10=0.0797 | Val HR@10=0.4800 | Val NDCG@10=0.1054
------------------------------------------------------------------------------------------
Epoch 03 | bpr_loss=0.3672 | Val RMSE=6.8283 | Val MAE=6.6173 | Val P@10=0.0810 | Val R@10=0.0776 | Val HR@10=0.4733 | Val NDCG@10=0.1031
------------------------------------------------------------------------------------------
Epoch 04 | bpr_loss=0.3671 | Val RMSE=6.8290 | Val MAE=6.6181 | Val P@10=0.0833 | Val R@10=0.0797 | Val HR@10=0.4867 | Val NDCG@10=0.1051
------------------------------------------------------------------------------------------
Epoch 05 | bpr_loss=0.3672 | Val RMSE=6.8288 | Val MAE=6.6180 | Val P@10=0.0817 | Va

## Recommendations, Metrics, Further Analysis

**Note:** Only the metrics for Matrix Factorization (BPR) was saved in a folder

### 1. Metric for Matrix Factorization with no bias (MSE)

In [25]:
# Metrics on Matrix Factorization model with no bias
print("=" * 80)
print("METRICS FOR MATRIX FACTORIZATION WITH NO BIAS")
print("=" * 80)
print("\n")
# 1. Regression Metrics
train_rmse_no_bias, train_mae_no_bias = evaluate_regression(model_no_bias, train_loader)
test_rmse_no_bias, test_mae_no_bias   = evaluate_regression(model_no_bias, test_loader)

print("=" * 80)
print("Regression Metrics (full datasets, all users)")
print(f"Train: {len(train_df):,} interactions | RMSE={train_rmse_no_bias:.4f}, MAE={train_mae_no_bias:.4f}")
print(f"Test: {len(test_df):,} interactions  | RMSE={test_rmse_no_bias:.4f},  MAE={test_mae_no_bias:.4f}")
print("=" * 80)

# 2. Ranking Metrics
# Using 300 users here; set to None for truly all users (very slow)
FULL_EVAL_USERS = 300   # increase this for more coverage; None = all users

test_p_no_bias, test_r_no_bias, test_hr_no_bias, test_ndcg_no_bias = evaluate_full_catalog_at_k(
    model=model_no_bias,
    pos_by_user=test_pos_by_user,
    seen_items_by_user=user_seen_train,
    k=EVAL_K,
    max_users=FULL_EVAL_USERS
)
train_p_no_bias, train_r_no_bias, train_hr_no_bias, train_ndcg_no_bias = evaluate_full_catalog_at_k(
    model=model_no_bias,
    pos_by_user=train_pos_by_user,
    seen_items_by_user={},   # no exclusions: model was trained on these
    k=EVAL_K,
    max_users=FULL_EVAL_USERS
)

print(f"\nRanking Metrics — Full Catalog @ k={EVAL_K}")
print(f"(users sampled: up to {FULL_EVAL_USERS}, positives = rating >= {POS_THRESHOLD})")
print(f"{'':20s} {'P@10':>8} {'R@10':>8} {'HR@10':>8} {'NDCG@10':>9}")
print(f"{'Train set':20s} {train_p_no_bias:>8.4f} {train_r_no_bias:>8.4f} {train_hr_no_bias:>8.4f} {train_ndcg_no_bias:>9.4f}")
print(f"{'Test set':20s} {test_p_no_bias:>8.4f} {test_r_no_bias:>8.4f} {test_hr_no_bias:>8.4f} {test_ndcg_no_bias:>9.4f}")

# 3. Segmented analysis (300 users, matches groupmate exactly)
ANALYSIS_MAX_USERS = 300
LONG_TAIL_QUANTILE = 0.80
PERSONALIZATION_SAMPLE_USERS = 200

# Sample 300 users for deep analysis
analysis_users = list(test_pos_by_user.keys())
np.random.seed(SEED)
if len(analysis_users) > ANALYSIS_MAX_USERS:
    analysis_users = list(np.random.choice(analysis_users, size=ANALYSIS_MAX_USERS, replace=False))

# Get their recommendations (full catalog)
def get_topk_recs_mf(model, user_ids, seen_items_by_user, k=10):
    model.eval()
    all_item_ids = np.arange(NUM_ITEMS_EMBEDDING)
    results = {}
    for user_id in user_ids:
        seen = seen_items_by_user.get(user_id, set())
        candidates = np.setdiff1d(all_item_ids, list(seen))
        if len(candidates) == 0:
            results[user_id] = []
            continue
        u_tensor = torch.tensor([user_id] * len(candidates), dtype=torch.long).to(DEVICE)
        i_tensor = torch.tensor(candidates, dtype=torch.long).to(DEVICE)
        with torch.no_grad():
            scores = model(u_tensor, i_tensor).cpu().numpy()
        top_idx = np.argsort(scores)[-k:][::-1]
        results[user_id] = candidates[top_idx].tolist()
    return results

def user_level_metrics_from_recs(recs_by_user, gt_by_user, k=10):
    rows = []
    for u, recs in recs_by_user.items():
        gt = gt_by_user.get(u, set())
        if len(gt) == 0:
            continue
        topk = recs[:k]
        hits = np.isin(topk, list(gt)).astype(np.float32)
        num_hits = float(hits.sum())
        denom = np.log2(np.arange(2, len(topk) + 2))
        dcg = float((hits / denom).sum())
        ideal_hits = np.ones(min(k, len(gt)), dtype=np.float32)
        idcg = float((ideal_hits / np.log2(np.arange(2, len(ideal_hits) + 2))).sum())
        rows.append({
            "user_idx": u,
            "precision": num_hits / k,
            "recall": num_hits / len(gt),
            "hitrate": 1.0 if num_hits > 0 else 0.0,
            "ndcg": (dcg / idcg) if idcg > 0 else 0.0,
            "num_test_positives": len(gt)
        })
    return pd.DataFrame(rows)

audit_recs = get_topk_recs_mf(model_no_bias, analysis_users, user_seen_train, k=EVAL_K)
audit_metrics_df = user_level_metrics_from_recs(audit_recs, test_pos_by_user, k=EVAL_K)

# 3.1 — Activity segment breakdown (identical to Two-Tower)
train_user_activity = train_df.groupby("user_idx").size().rename("train_interactions")
train_user_activity = train_user_activity.reindex(range(NUM_USERS_EMBEDDING), fill_value=0)

activity_df = pd.DataFrame({
    "user_idx": analysis_users,
    "train_interactions": [train_user_activity.get(u, 0) for u in analysis_users]
})
activity_df["activity_segment"] = pd.qcut(
    activity_df["train_interactions"].rank(method="first"),
    q=4, labels=["light", "medium", "heavy", "power"], duplicates="drop"
)
audit_metrics_df = audit_metrics_df.merge(activity_df, on="user_idx", how="left")

segment_perf = audit_metrics_df.groupby("activity_segment", observed=False)[["precision","recall","hitrate","ndcg"]].mean()
segment_counts = audit_metrics_df.groupby("activity_segment", observed=False).size().rename("num_users")
print(f"\n=== Performance by User Activity Segment (n={ANALYSIS_MAX_USERS} users) ===")
print(pd.concat([segment_perf, segment_counts], axis=1).round(4))

# 3.2 — Popularity bucket hit rate (identical to Two-Tower)
item_pop = train_df["anime_idx"].value_counts().rename("train_popularity")
item_pop = item_pop.reindex(range(NUM_ITEMS_EMBEDDING), fill_value=0)
nonzero_item_pop = item_pop[item_pop.index != 0]
pop_threshold_val = nonzero_item_pop.quantile(LONG_TAIL_QUANTILE)
item_bucket = pd.Series(index=nonzero_item_pop.index, dtype="object")
item_bucket[nonzero_item_pop >= pop_threshold_val] = "head_mid"
item_bucket[nonzero_item_pop < pop_threshold_val]  = "long_tail"
item_novelty = -np.log(nonzero_item_pop / nonzero_item_pop.sum() + 1e-12)

bucket_hit_rows = []
for u, recs in audit_recs.items():
    gt = test_pos_by_user.get(u, set())
    topk = set(recs[:EVAL_K])
    for item in gt:
        bucket_hit_rows.append({
            "user_idx": u, "anime_idx": item,
            "popularity_bucket": item_bucket.get(item, "head_mid"),
            "hit_at_k": 1 if item in topk else 0
        })

bucket_hit_df = pd.DataFrame(bucket_hit_rows)
print(f"\n=== Hit Rate by Ground-Truth Popularity Bucket (n={ANALYSIS_MAX_USERS} users) ===")
pop_bucket_perf   = bucket_hit_df.groupby("popularity_bucket")["hit_at_k"].mean()
pop_bucket_counts = bucket_hit_df.groupby("popularity_bucket").size().rename("num_items")
print(pd.concat([pop_bucket_perf.rename("hit_rate"), pop_bucket_counts], axis=1).round(4))

# 3.3 — Bias / coverage summary (identical to Two-Tower)
all_recommended_items = [item for recs in audit_recs.values() for item in recs[:EVAL_K]]
rec_counter = Counter(all_recommended_items)
rec_freq  = np.array(list(rec_counter.values()), dtype=np.float64)
rec_share = rec_freq / rec_freq.sum()
all_gt_items = [item for u in analysis_users for item in test_pos_by_user.get(u, set())]

avg_rec_pop = float(np.mean([item_pop.get(i, 0) for i in all_recommended_items]))
avg_gt_pop  = float(np.mean([item_pop.get(i, 0) for i in all_gt_items])) if all_gt_items else 0.0
rec_lt_share = float(np.mean([1.0 if item_bucket.get(i,"head_mid")=="long_tail" else 0.0 for i in all_recommended_items]))
gt_lt_share  = float(np.mean([1.0 if item_bucket.get(i,"head_mid")=="long_tail" else 0.0 for i in all_gt_items])) if all_gt_items else 0.0

sample_users = list(audit_recs.keys())
if len(sample_users) > PERSONALIZATION_SAMPLE_USERS:
    np.random.seed(SEED)
    sample_users = list(np.random.choice(sample_users, size=PERSONALIZATION_SAMPLE_USERS, replace=False))
jaccards = []
for i in range(len(sample_users)):
    set_i = set(audit_recs[sample_users[i]][:EVAL_K])
    for j in range(i + 1, len(sample_users)):
        set_j = set(audit_recs[sample_users[j]][:EVAL_K])
        union = len(set_i | set_j)
        if union > 0:
            jaccards.append(len(set_i & set_j) / union)
personalization = 1.0 - (float(np.mean(jaccards)) if jaccards else 0.0)

summary_df = pd.DataFrame({
    "metric": ["catalog_coverage_at_k", "recommendation_hhi",
               "avg_recommended_item_popularity", "avg_ground_truth_item_popularity",
               "popularity_gap_rec_minus_truth", "recommended_long_tail_share",
               "ground_truth_long_tail_share", "long_tail_gap_rec_minus_truth",
               "avg_recommendation_novelty", "personalization_1_minus_avg_jaccard"],
    "value": [len(rec_counter)/(NUM_ITEMS_EMBEDDING-1), float(np.sum(rec_share**2)),
              avg_rec_pop, avg_gt_pop, avg_rec_pop - avg_gt_pop,
              rec_lt_share, gt_lt_share, rec_lt_share - gt_lt_share,
              float(np.mean([item_novelty.get(i,0.0) for i in all_recommended_items])),
              personalization]
})
print(f"\n=== Recommendation Bias / Coverage Summary (n={ANALYSIS_MAX_USERS} users) ===")
print(summary_df.round(4))

METRICS FOR MATRIX FACTORIZATION WITH NO BIAS


Regression Metrics (full datasets, all users)
Train: 49,742,364 interactions | RMSE=1.6133, MAE=1.1705
Test: 6,215,263 interactions  | RMSE=1.6394,  MAE=1.1872

Ranking Metrics — Full Catalog @ k=10
(users sampled: up to 300, positives = rating >= 6.0)
                         P@10     R@10    HR@10   NDCG@10
Train set              0.2790   0.0540   0.8333    0.2979
Test set               0.0527   0.0645   0.3800    0.0665

=== Performance by User Activity Segment (n=300 users) ===
                  precision  recall  hitrate    ndcg  num_users
activity_segment                                               
light                0.0373  0.1069   0.3067  0.0663         75
medium               0.0333  0.0573   0.3200  0.0431         75
heavy                0.0613  0.0635   0.4400  0.0702         75
power                0.0787  0.0302   0.4533  0.0864         75

=== Hit Rate by Ground-Truth Popularity Bucket (n=300 users) ===
               

In [26]:
print("=" * 80)
print("METRICS FOR MATRIX FACTORIZATION WITH NO BIAS")
print("=" * 80)
print("\n")

test_p_no_bias, test_r_no_bias, test_hr_no_bias, test_ndcg_no_bias = evaluate_ranking_at_k(
    model=model_no_bias,
    pos_by_user=test_pos_by_user,
    seen_items_by_user=user_seen_train,
    k=EVAL_K,
    max_users=EVAL_MAX_USERS,
    neg_samples=EVAL_NEG_SAMPLES
)

print("Final Test Metrics:")
print(f"P@{EVAL_K}   = {test_p_no_bias:.4f}")
print(f"R@{EVAL_K}   = {test_r_no_bias:.4f}")
print(f"HR@{EVAL_K}  = {test_hr_no_bias:.4f}")
print(f"NDCG@{EVAL_K}= {test_ndcg_no_bias:.4f}")

METRICS FOR MATRIX FACTORIZATION WITH NO BIAS


Final Test Metrics:
P@10   = 0.3970
R@10   = 0.5111
HR@10  = 0.9533
NDCG@10= 0.5626


In [27]:
OUTPUT_PATH = Path("/content/drive/MyDrive/matrix_factorization_analysis_outputs_full_catalog")
analysis_output_dir = BASE_PATH / "mf_threshold_6"
analysis_output_dir.mkdir(parents=True, exist_ok=True)

audit_metrics_df.to_csv(analysis_output_dir / "mf_no_bias_user_level_audit_metrics.csv", index=False)
summary_df.to_csv(analysis_output_dir / "mf_no_bias_recommendation_bias_coverage_summary.csv", index=False)
pd.concat([segment_perf, segment_counts], axis=1).reset_index().to_csv(
    analysis_output_dir / "mf_no_bias_activity_segment_performance.csv", index=False
)
if len(bucket_hit_df) > 0:
    bucket_hit_df.to_csv(analysis_output_dir / "mf_no_bias_ground_truth_popularity_hit_analysis.csv", index=False)

print("\nSaved analysis files to:", analysis_output_dir)


Saved analysis files to: /content/drive/MyDrive/yuran_files/mf_threshold_6


### 2. Metrics for Matrix Factorization with Biases (MSE)

In [28]:
# Metrics on Matrix Factorization model with no bias
print("=" * 80)
print("METRICS FOR MATRIX FACTORIZATION WITH BIAS")
print("=" * 80)
print("\n")

# 1. Regression metrics (entire train and test sets, no sampling)
train_rmse, train_mae = evaluate_regression(model, train_loader)
test_rmse, test_mae   = evaluate_regression(model, test_loader)

print("=" * 80)
print("Regression Metrics (full datasets, all users)")
print(f"Train: {len(train_df):,} interactions | RMSE={train_rmse:.4f}, MAE={train_mae:.4f}")
print(f"Test: {len(test_df):,} interactions  | RMSE={test_rmse:.4f},  MAE={test_mae:.4f}")
print("=" * 80)

# 2. Full ranking metrics (larger user sample = "full" report numbers)
# Using 300 users here; set to None for truly all users (very slow)
FULL_EVAL_USERS = 300   # increase this for more coverage; None = all users

test_p, test_r, test_hr, test_ndcg = evaluate_full_catalog_at_k(
    model=model,
    pos_by_user=test_pos_by_user,
    seen_items_by_user=user_seen_train,
    k=EVAL_K,
    max_users=FULL_EVAL_USERS
)
train_p, train_r, train_hr, train_ndcg = evaluate_full_catalog_at_k(
    model=model,
    pos_by_user=train_pos_by_user,
    seen_items_by_user={},   # no exclusions: model was trained on these
    k=EVAL_K,
    max_users=FULL_EVAL_USERS
)

print(f"\nRanking Metrics — Full Catalog @ k={EVAL_K}")
print(f"(users sampled: up to {FULL_EVAL_USERS}, positives = rating >= {POS_THRESHOLD})")
print(f"{'':20s} {'P@10':>8} {'R@10':>8} {'HR@10':>8} {'NDCG@10':>9}")
print(f"{'Train set':20s} {train_p:>8.4f} {train_r:>8.4f} {train_hr:>8.4f} {train_ndcg:>9.4f}")
print(f"{'Test set':20s} {test_p:>8.4f} {test_r:>8.4f} {test_hr:>8.4f} {test_ndcg:>9.4f}")

# 3. Segmented analysis (300 users, matches groupmate exactly)
ANALYSIS_MAX_USERS = 300
LONG_TAIL_QUANTILE = 0.80
PERSONALIZATION_SAMPLE_USERS = 200

# Sample 300 users for deep analysis
analysis_users = list(test_pos_by_user.keys())
np.random.seed(SEED)
if len(analysis_users) > ANALYSIS_MAX_USERS:
    analysis_users = list(np.random.choice(analysis_users, size=ANALYSIS_MAX_USERS, replace=False))

# Get their recommendations (full catalog)
def get_topk_recs_mf(model, user_ids, seen_items_by_user, k=10):
    model.eval()
    all_item_ids = np.arange(NUM_ITEMS_EMBEDDING)
    results = {}
    for user_id in user_ids:
        seen = seen_items_by_user.get(user_id, set())
        candidates = np.setdiff1d(all_item_ids, list(seen))
        if len(candidates) == 0:
            results[user_id] = []
            continue
        u_tensor = torch.tensor([user_id] * len(candidates), dtype=torch.long).to(DEVICE)
        i_tensor = torch.tensor(candidates, dtype=torch.long).to(DEVICE)
        with torch.no_grad():
            scores = model(u_tensor, i_tensor).cpu().numpy()
        top_idx = np.argsort(scores)[-k:][::-1]
        results[user_id] = candidates[top_idx].tolist()
    return results

def user_level_metrics_from_recs(recs_by_user, gt_by_user, k=10):
    rows = []
    for u, recs in recs_by_user.items():
        gt = gt_by_user.get(u, set())
        if len(gt) == 0:
            continue
        topk = recs[:k]
        hits = np.isin(topk, list(gt)).astype(np.float32)
        num_hits = float(hits.sum())
        denom = np.log2(np.arange(2, len(topk) + 2))
        dcg = float((hits / denom).sum())
        ideal_hits = np.ones(min(k, len(gt)), dtype=np.float32)
        idcg = float((ideal_hits / np.log2(np.arange(2, len(ideal_hits) + 2))).sum())
        rows.append({
            "user_idx": u,
            "precision": num_hits / k,
            "recall": num_hits / len(gt),
            "hitrate": 1.0 if num_hits > 0 else 0.0,
            "ndcg": (dcg / idcg) if idcg > 0 else 0.0,
            "num_test_positives": len(gt)
        })
    return pd.DataFrame(rows)

audit_recs = get_topk_recs_mf(model, analysis_users, user_seen_train, k=EVAL_K)
audit_metrics_df = user_level_metrics_from_recs(audit_recs, test_pos_by_user, k=EVAL_K)

# 3.1 — Activity segment breakdown (identical to Two-Tower)
train_user_activity = train_df.groupby("user_idx").size().rename("train_interactions")
train_user_activity = train_user_activity.reindex(range(NUM_USERS_EMBEDDING), fill_value=0)

activity_df = pd.DataFrame({
    "user_idx": analysis_users,
    "train_interactions": [train_user_activity.get(u, 0) for u in analysis_users]
})
activity_df["activity_segment"] = pd.qcut(
    activity_df["train_interactions"].rank(method="first"),
    q=4, labels=["light", "medium", "heavy", "power"], duplicates="drop"
)
audit_metrics_df = audit_metrics_df.merge(activity_df, on="user_idx", how="left")

segment_perf   = audit_metrics_df.groupby("activity_segment", observed=False)[["precision","recall","hitrate","ndcg"]].mean()
segment_counts = audit_metrics_df.groupby("activity_segment", observed=False).size().rename("num_users")
print(f"\n=== Performance by User Activity Segment (n={ANALYSIS_MAX_USERS} users) ===")
print(pd.concat([segment_perf, segment_counts], axis=1).round(4))

# 3.2 — Popularity bucket hit rate (identical to Two-Tower)
item_pop = train_df["anime_idx"].value_counts().rename("train_popularity")
item_pop = item_pop.reindex(range(NUM_ITEMS_EMBEDDING), fill_value=0)
nonzero_item_pop = item_pop[item_pop.index != 0]
pop_threshold_val = nonzero_item_pop.quantile(LONG_TAIL_QUANTILE)
item_bucket = pd.Series(index=nonzero_item_pop.index, dtype="object")
item_bucket[nonzero_item_pop >= pop_threshold_val] = "head_mid"
item_bucket[nonzero_item_pop < pop_threshold_val]  = "long_tail"
item_novelty = -np.log(nonzero_item_pop / nonzero_item_pop.sum() + 1e-12)

bucket_hit_rows = []
for u, recs in audit_recs.items():
    gt = test_pos_by_user.get(u, set())
    topk = set(recs[:EVAL_K])
    for item in gt:
        bucket_hit_rows.append({
            "user_idx": u, "anime_idx": item,
            "popularity_bucket": item_bucket.get(item, "head_mid"),
            "hit_at_k": 1 if item in topk else 0
        })

bucket_hit_df = pd.DataFrame(bucket_hit_rows)
print(f"\n=== Hit Rate by Ground-Truth Popularity Bucket (n={ANALYSIS_MAX_USERS} users) ===")
pop_bucket_perf   = bucket_hit_df.groupby("popularity_bucket")["hit_at_k"].mean()
pop_bucket_counts = bucket_hit_df.groupby("popularity_bucket").size().rename("num_items")
print(pd.concat([pop_bucket_perf.rename("hit_rate"), pop_bucket_counts], axis=1).round(4))

# 3.3 — Bias / coverage summary (identical to Two-Tower)
all_recommended_items = [item for recs in audit_recs.values() for item in recs[:EVAL_K]]
rec_counter = Counter(all_recommended_items)
rec_freq  = np.array(list(rec_counter.values()), dtype=np.float64)
rec_share = rec_freq / rec_freq.sum()
all_gt_items = [item for u in analysis_users for item in test_pos_by_user.get(u, set())]

avg_rec_pop = float(np.mean([item_pop.get(i, 0) for i in all_recommended_items]))
avg_gt_pop  = float(np.mean([item_pop.get(i, 0) for i in all_gt_items])) if all_gt_items else 0.0
rec_lt_share = float(np.mean([1.0 if item_bucket.get(i,"head_mid")=="long_tail" else 0.0 for i in all_recommended_items]))
gt_lt_share  = float(np.mean([1.0 if item_bucket.get(i,"head_mid")=="long_tail" else 0.0 for i in all_gt_items])) if all_gt_items else 0.0

sample_users = list(audit_recs.keys())
if len(sample_users) > PERSONALIZATION_SAMPLE_USERS:
    np.random.seed(SEED)
    sample_users = list(np.random.choice(sample_users, size=PERSONALIZATION_SAMPLE_USERS, replace=False))
jaccards = []
for i in range(len(sample_users)):
    set_i = set(audit_recs[sample_users[i]][:EVAL_K])
    for j in range(i + 1, len(sample_users)):
        set_j = set(audit_recs[sample_users[j]][:EVAL_K])
        union = len(set_i | set_j)
        if union > 0:
            jaccards.append(len(set_i & set_j) / union)
personalization = 1.0 - (float(np.mean(jaccards)) if jaccards else 0.0)

summary_df = pd.DataFrame({
    "metric": ["catalog_coverage_at_k", "recommendation_hhi",
               "avg_recommended_item_popularity", "avg_ground_truth_item_popularity",
               "popularity_gap_rec_minus_truth", "recommended_long_tail_share",
               "ground_truth_long_tail_share", "long_tail_gap_rec_minus_truth",
               "avg_recommendation_novelty", "personalization_1_minus_avg_jaccard"],
    "value": [len(rec_counter)/(NUM_ITEMS_EMBEDDING-1), float(np.sum(rec_share**2)),
              avg_rec_pop, avg_gt_pop, avg_rec_pop - avg_gt_pop,
              rec_lt_share, gt_lt_share, rec_lt_share - gt_lt_share,
              float(np.mean([item_novelty.get(i,0.0) for i in all_recommended_items])),
              personalization]
})
print(f"\n=== Recommendation Bias / Coverage Summary (n={ANALYSIS_MAX_USERS} users) ===")
print(summary_df.round(4))

METRICS FOR MATRIX FACTORIZATION WITH BIAS


Regression Metrics (full datasets, all users)
Train: 49,742,364 interactions | RMSE=1.2765, MAE=0.9544
Test: 6,215,263 interactions  | RMSE=1.2882,  MAE=0.9641

Ranking Metrics — Full Catalog @ k=10
(users sampled: up to 300, positives = rating >= 6.0)
                         P@10     R@10    HR@10   NDCG@10
Train set              0.2170   0.0275   0.7300    0.2334
Test set               0.0380   0.0315   0.2500    0.0452

=== Performance by User Activity Segment (n=300 users) ===
                  precision  recall  hitrate    ndcg  num_users
activity_segment                                               
light                0.0160  0.0466   0.1200  0.0292         75
medium               0.0173  0.0231   0.1600  0.0235         75
heavy                0.0373  0.0289   0.2667  0.0413         75
power                0.0813  0.0272   0.4533  0.0867         75

=== Hit Rate by Ground-Truth Popularity Bucket (n=300 users) ===
                  

In [29]:
# Metrics on Matrix Factorization model with bias
print("=" * 80)
print("METRICS FOR MATRIX FACTORIZATION WITH BIAS")
print("=" * 80)
print("\n")

test_p_with_bias, test_r_with_bias, test_hr_with_bias, test_ndcg_with_bias = evaluate_ranking_at_k(
    model=model,
    pos_by_user=test_pos_by_user,
    seen_items_by_user=user_seen_train,
    k=EVAL_K,
    max_users=EVAL_MAX_USERS,
    neg_samples=EVAL_NEG_SAMPLES
)

print("\nFinal Test Metrics:")
print(f"P@{EVAL_K}   = {test_p_with_bias:.4f}")
print(f"R@{EVAL_K}   = {test_r_with_bias:.4f}")
print(f"HR@{EVAL_K}  = {test_hr_with_bias:.4f}")
print(f"NDCG@{EVAL_K}= {test_ndcg_with_bias:.4f}")

METRICS FOR MATRIX FACTORIZATION WITH BIAS



Final Test Metrics:
P@10   = 0.3337
R@10   = 0.3742
HR@10  = 0.8933
NDCG@10= 0.4498


In [ ]:
OUTPUT_PATH = Path("/content/drive/MyDrive/matrix_factorization_analysis_outputs_full_catalog")
analysis_output_dir = BASE_PATH / "mf_threshold_6"
# analysis_output_dir.mkdir(parents=True, exist_ok=True)

audit_metrics_df.to_csv(analysis_output_dir / "mf_bias_user_level_audit_metrics.csv", index=False)
summary_df.to_csv(analysis_output_dir / "mf_bias_recommendation_bias_coverage_summary.csv", index=False)
pd.concat([segment_perf, segment_counts], axis=1).reset_index().to_csv(
    analysis_output_dir / "mf_bias_activity_segment_performance.csv", index=False
)
if len(bucket_hit_df) > 0:
    bucket_hit_df.to_csv(analysis_output_dir / "mf_bias_ground_truth_popularity_hit_analysis.csv", index=False)

print("\nSaved analysis files to:", analysis_output_dir)


Saved analysis files to: /content/drive/MyDrive/yuran_files/mf_threshold_6


### 3. Metrics for Matrix Factorization model with no bias (BPR)

In [31]:
# Metrics on Matrix Factorization model with no bias (BPR)
print("=" * 80)
print("METRICS FOR BPR MATRIX FACTORIZATION WITH NO BIAS")
print("=" * 80)
print("\n")
# 1. Regression Metrics
train_rmse_no_bias_bpr, train_mae_no_bias_bpr = evaluate_regression(model_no_bias_bpr, bpr_train_loader)
# test_rmse_no_bias_bpr, test_mae_no_bias_bpr = evaluate_regression(model_no_bias_bpr, test_loader)

print("=" * 80)
print("Regression Metrics (full datasets, all users)")
print(f"Train: {len(train_df):,} interactions | RMSE={train_rmse_no_bias_bpr:.4f}, MAE={train_mae_no_bias_bpr:.4f}")
# print(f"Test: {len(test_df):,} interactions  | RMSE={test_rmse_no_bias_bpr:.4f},  MAE={test_mae_no_bias_bpr:.4f}")
print("=" * 80)

# 2. Ranking Metrics
# Using 300 users here; set to None for truly all users (very slow)
FULL_EVAL_USERS = 300   # increase this for more coverage; None = all users

test_p_no_bias_bpr, test_r_no_bias_bpr, test_hr_no_bias_bpr, test_ndcg_no_bias_bpr = evaluate_full_catalog_at_k(
    model=model_no_bias_bpr,
    pos_by_user=test_pos_by_user,
    seen_items_by_user=user_seen_train,
    k=EVAL_K,
    max_users=FULL_EVAL_USERS
)
train_p_no_bias_bpr, train_r_no_bias_bpr, train_hr_no_bias_bpr, train_ndcg_no_bias_bpr = evaluate_full_catalog_at_k(
    model=model_no_bias_bpr,
    pos_by_user=train_pos_by_user,
    seen_items_by_user={},   # no exclusions: model was trained on these
    k=EVAL_K,
    max_users=FULL_EVAL_USERS
)

print(f"\nRanking Metrics — Full Catalog @ k={EVAL_K}")
print(f"(users sampled: up to {FULL_EVAL_USERS}, positives = rating >= {POS_THRESHOLD})")
print(f"{'':20s} {'P@10':>8} {'R@10':>8} {'HR@10':>8} {'NDCG@10':>9}")
print(f"{'Train set':20s} {train_p_no_bias_bpr:>8.4f} {train_r_no_bias_bpr:>8.4f} {train_hr_no_bias_bpr:>8.4f} {train_ndcg_no_bias_bpr:>9.4f}")
print(f"{'Test set':20s} {test_p_no_bias_bpr:>8.4f} {test_r_no_bias_bpr:>8.4f} {test_hr_no_bias_bpr:>8.4f} {test_ndcg_no_bias_bpr:>9.4f}")

# 3. Segmented analysis (300 users, matches groupmate exactly)
ANALYSIS_MAX_USERS = 300
LONG_TAIL_QUANTILE = 0.80
PERSONALIZATION_SAMPLE_USERS = 200

# Sample 300 users for deep analysis
analysis_users = list(test_pos_by_user.keys())
np.random.seed(SEED)
if len(analysis_users) > ANALYSIS_MAX_USERS:
    analysis_users = list(np.random.choice(analysis_users, size=ANALYSIS_MAX_USERS, replace=False))

# Get their recommendations (full catalog)
def get_topk_recs_mf(model, user_ids, seen_items_by_user, k=10):
    model.eval()
    all_item_ids = np.arange(NUM_ITEMS_EMBEDDING)
    results = {}
    for user_id in user_ids:
        seen = seen_items_by_user.get(user_id, set())
        candidates = np.setdiff1d(all_item_ids, list(seen))
        if len(candidates) == 0:
            results[user_id] = []
            continue
        u_tensor = torch.tensor([user_id] * len(candidates), dtype=torch.long).to(DEVICE)
        i_tensor = torch.tensor(candidates, dtype=torch.long).to(DEVICE)
        with torch.no_grad():
            scores = model(u_tensor, i_tensor).cpu().numpy()
        top_idx = np.argsort(scores)[-k:][::-1]
        results[user_id] = candidates[top_idx].tolist()
    return results

def user_level_metrics_from_recs(recs_by_user, gt_by_user, k=10):
    rows = []
    for u, recs in recs_by_user.items():
        gt = gt_by_user.get(u, set())
        if len(gt) == 0:
            continue
        topk = recs[:k]
        hits = np.isin(topk, list(gt)).astype(np.float32)
        num_hits = float(hits.sum())
        denom = np.log2(np.arange(2, len(topk) + 2))
        dcg = float((hits / denom).sum())
        ideal_hits = np.ones(min(k, len(gt)), dtype=np.float32)
        idcg = float((ideal_hits / np.log2(np.arange(2, len(ideal_hits) + 2))).sum())
        rows.append({
            "user_idx": u,
            "precision": num_hits / k,
            "recall": num_hits / len(gt),
            "hitrate": 1.0 if num_hits > 0 else 0.0,
            "ndcg": (dcg / idcg) if idcg > 0 else 0.0,
            "num_test_positives": len(gt)
        })
    return pd.DataFrame(rows)

audit_recs = get_topk_recs_mf(model_no_bias_bpr, analysis_users, user_seen_train, k=EVAL_K)
audit_metrics_df = user_level_metrics_from_recs(audit_recs, test_pos_by_user, k=EVAL_K)

# 3.1 — Activity segment breakdown (identical to Two-Tower)
train_user_activity = train_df.groupby("user_idx").size().rename("train_interactions")
train_user_activity = train_user_activity.reindex(range(NUM_USERS_EMBEDDING), fill_value=0)

activity_df = pd.DataFrame({
    "user_idx": analysis_users,
    "train_interactions": [train_user_activity.get(u, 0) for u in analysis_users]
})
activity_df["activity_segment"] = pd.qcut(
    activity_df["train_interactions"].rank(method="first"),
    q=4, labels=["light", "medium", "heavy", "power"], duplicates="drop"
)
audit_metrics_df = audit_metrics_df.merge(activity_df, on="user_idx", how="left")

segment_perf = audit_metrics_df.groupby("activity_segment", observed=False)[["precision","recall","hitrate","ndcg"]].mean()
segment_counts = audit_metrics_df.groupby("activity_segment", observed=False).size().rename("num_users")
print(f"\n=== Performance by User Activity Segment (n={ANALYSIS_MAX_USERS} users) ===")
print(pd.concat([segment_perf, segment_counts], axis=1).round(4))

# 3.2 — Popularity bucket hit rate (identical to Two-Tower)
item_pop = train_df["anime_idx"].value_counts().rename("train_popularity")
item_pop = item_pop.reindex(range(NUM_ITEMS_EMBEDDING), fill_value=0)
nonzero_item_pop = item_pop[item_pop.index != 0]
pop_threshold_val = nonzero_item_pop.quantile(LONG_TAIL_QUANTILE)
item_bucket = pd.Series(index=nonzero_item_pop.index, dtype="object")
item_bucket[nonzero_item_pop >= pop_threshold_val] = "head_mid"
item_bucket[nonzero_item_pop < pop_threshold_val]  = "long_tail"
item_novelty = -np.log(nonzero_item_pop / nonzero_item_pop.sum() + 1e-12)

bucket_hit_rows = []
for u, recs in audit_recs.items():
    gt = test_pos_by_user.get(u, set())
    topk = set(recs[:EVAL_K])
    for item in gt:
        bucket_hit_rows.append({
            "user_idx": u, "anime_idx": item,
            "popularity_bucket": item_bucket.get(item, "head_mid"),
            "hit_at_k": 1 if item in topk else 0
        })

bucket_hit_df = pd.DataFrame(bucket_hit_rows)
print(f"\n=== Hit Rate by Ground-Truth Popularity Bucket (n={ANALYSIS_MAX_USERS} users) ===")
pop_bucket_perf   = bucket_hit_df.groupby("popularity_bucket")["hit_at_k"].mean()
pop_bucket_counts = bucket_hit_df.groupby("popularity_bucket").size().rename("num_items")
print(pd.concat([pop_bucket_perf.rename("hit_rate"), pop_bucket_counts], axis=1).round(4))

# 3.3 — Bias / coverage summary (identical to Two-Tower)
all_recommended_items = [item for recs in audit_recs.values() for item in recs[:EVAL_K]]
rec_counter = Counter(all_recommended_items)
rec_freq  = np.array(list(rec_counter.values()), dtype=np.float64)
rec_share = rec_freq / rec_freq.sum()
all_gt_items = [item for u in analysis_users for item in test_pos_by_user.get(u, set())]

avg_rec_pop = float(np.mean([item_pop.get(i, 0) for i in all_recommended_items]))
avg_gt_pop  = float(np.mean([item_pop.get(i, 0) for i in all_gt_items])) if all_gt_items else 0.0
rec_lt_share = float(np.mean([1.0 if item_bucket.get(i,"head_mid")=="long_tail" else 0.0 for i in all_recommended_items]))
gt_lt_share  = float(np.mean([1.0 if item_bucket.get(i,"head_mid")=="long_tail" else 0.0 for i in all_gt_items])) if all_gt_items else 0.0

sample_users = list(audit_recs.keys())
if len(sample_users) > PERSONALIZATION_SAMPLE_USERS:
    np.random.seed(SEED)
    sample_users = list(np.random.choice(sample_users, size=PERSONALIZATION_SAMPLE_USERS, replace=False))
jaccards = []
for i in range(len(sample_users)):
    set_i = set(audit_recs[sample_users[i]][:EVAL_K])
    for j in range(i + 1, len(sample_users)):
        set_j = set(audit_recs[sample_users[j]][:EVAL_K])
        union = len(set_i | set_j)
        if union > 0:
            jaccards.append(len(set_i & set_j) / union)
personalization = 1.0 - (float(np.mean(jaccards)) if jaccards else 0.0)

summary_df = pd.DataFrame({
    "metric": ["catalog_coverage_at_k", "recommendation_hhi",
               "avg_recommended_item_popularity", "avg_ground_truth_item_popularity",
               "popularity_gap_rec_minus_truth", "recommended_long_tail_share",
               "ground_truth_long_tail_share", "long_tail_gap_rec_minus_truth",
               "avg_recommendation_novelty", "personalization_1_minus_avg_jaccard"],
    "value": [len(rec_counter)/(NUM_ITEMS_EMBEDDING-1), float(np.sum(rec_share**2)),
              avg_rec_pop, avg_gt_pop, avg_rec_pop - avg_gt_pop,
              rec_lt_share, gt_lt_share, rec_lt_share - gt_lt_share,
              float(np.mean([item_novelty.get(i,0.0) for i in all_recommended_items])),
              personalization]
})
print(f"\n=== Recommendation Bias / Coverage Summary (n={ANALYSIS_MAX_USERS} users) ===")
print(summary_df.round(4))

METRICS FOR BPR MATRIX FACTORIZATION WITH NO BIAS


Regression Metrics (full datasets, all users)
Train: 49,742,364 interactions | RMSE=7523.1826, MAE=6516.7433

Ranking Metrics — Full Catalog @ k=10
(users sampled: up to 300, positives = rating >= 6.0)
                         P@10     R@10    HR@10   NDCG@10
Train set              0.3580   0.0555   0.8700    0.3822
Test set               0.0737   0.0768   0.4700    0.0985

=== Performance by User Activity Segment (n=300 users) ===
                  precision  recall  hitrate    ndcg  num_users
activity_segment                                               
light                0.0387  0.1371   0.3200  0.1007         75
medium               0.0387  0.0516   0.3067  0.0615         75
heavy                0.0853  0.0697   0.5067  0.0953         75
power                0.1320  0.0489   0.7467  0.1366         75

=== Hit Rate by Ground-Truth Popularity Bucket (n=300 users) ===
                   hit_rate  num_items
popularity_bucket      

In [32]:
# Metrics on Matrix Factorization model with no bias (BPR)
print("=" * 80)
print("METRICS FOR BPR MATRIX FACTORIZATION WITH NO BIAS")
print("=" * 80)
print("\n")

test_p_bpr, test_r_bpr, test_hr_bpr, test_ndcg_bpr = evaluate_ranking_at_k(
    model=model_no_bias_bpr,
    pos_by_user=test_pos_by_user,
    seen_items_by_user=user_seen_train,
    k=EVAL_K,
    max_users=EVAL_MAX_USERS,
    neg_samples=EVAL_NEG_SAMPLES
)

print("\nFinal Test Metrics:")
print(f"P@{EVAL_K}   = {test_p_bpr:.4f}")
print(f"R@{EVAL_K}   = {test_r_bpr:.4f}")
print(f"HR@{EVAL_K}  = {test_hr_bpr:.4f}")
print(f"NDCG@{EVAL_K}= {test_ndcg_bpr:.4f}")

METRICS FOR BPR MATRIX FACTORIZATION WITH NO BIAS



Final Test Metrics:
P@10   = 0.4197
R@10   = 0.5037
HR@10  = 0.9300
NDCG@10= 0.5876


In [ ]:
OUTPUT_PATH = Path("/content/drive/MyDrive/matrix_factorization_analysis_outputs_full_catalog")

analysis_output_dir = OUTPUT_PATH / "mf_threshold_6"
# analysis_output_dir.mkdir(parents=True, exist_ok=True)

audit_metrics_df.to_csv(analysis_output_dir / "mf_bpr_user_level_audit_metrics.csv", index=False)
summary_df.to_csv(analysis_output_dir / "mf_bpr_recommendation_bias_coverage_summary.csv", index=False)
pd.concat([segment_perf, segment_counts], axis=1).reset_index().to_csv(
    analysis_output_dir / "mf_bpr_activity_segment_performance.csv", index=False
)
if len(bucket_hit_df) > 0:
    bucket_hit_df.to_csv(analysis_output_dir / "mf_bpr_ground_truth_popularity_hit_analysis.csv", index=False)

print("\nSaved analysis files to:", analysis_output_dir)


Saved analysis files to: /content/drive/MyDrive/matrix_factorization_analysis_outputs_full_catalog/mf_threshold_6
